# Chapter 13: Privacy, Governance & Identity Resolution - Code Validation

This notebook tests the code examples from Chapter 13 to ensure they are production-quality and executable.

## Table of Contents
1. Privacy-Compliant Identity Graph Schema (Code Listing 13.1)
2. Differential Privacy Budget Tracking and Noise Addition (Code Listing 13.2)

## 1. Privacy-Compliant Identity Graph Schema

Testing Code Listing 13.1: the simplified identity graph schema (nodes are pseudonymized
tokens, edges are typed, consent-scoped links between them) referenced in Section 2.4.

In [1]:
# Simplified identity graph schema
# Each node is a pseudonymized token; edges represent link types

identity_graph = {
    "nodes": [
        {"token": "t_abc123", "type": "email_hash", "created": "2024-01-15"},
        {"token": "t_def456", "type": "device_id", "created": "2024-01-15"},
        {"token": "t_ghi789", "type": "loyalty_id", "created": "2024-01-16"},
    ],
    "edges": [
        {
            "source": "t_abc123",
            "target": "t_def456",
            "link_type": "login",  # Deterministic: user logged in on this device
            "confidence": 1.0,
            "consent_basis": "legitimate_interest",
            "created": "2024-01-15",
            "expires": "2025-01-15"  # Links expire; require refresh
        },
        {
            "source": "t_abc123",
            "target": "t_ghi789",
            "link_type": "loyalty_enrollment",
            "confidence": 1.0,
            "consent_basis": "contract",  # Loyalty program terms
            "created": "2024-01-16",
            "expires": "2026-01-16"
        }
    ]
}

In [2]:
# --- Test: schema shape and consent-filtering query pattern ---
assert len(identity_graph["nodes"]) == 3
assert len(identity_graph["edges"]) == 2
assert all("consent_basis" in e for e in identity_graph["edges"])

# Consent filtering: only follow edges with a valid consent basis for the current purpose
allowed_bases = {"legitimate_interest", "contract"}
filtered_edges = [e for e in identity_graph["edges"] if e["consent_basis"] in allowed_bases]
assert len(filtered_edges) == 2
print(f"identity_graph has {len(identity_graph['nodes'])} nodes and "
      f"{len(identity_graph['edges'])} edges, all with a valid consent_basis")

identity_graph has 3 nodes and 2 edges, all with a valid consent_basis


## 2. Differential Privacy Budget Tracking and Noise Addition

Testing Code Listing 13.2: `PrivacyBudget` (per-party epsilon accounting against a daily
cap) and `DPNoiseAdder` (Laplace-mechanism noise addition for clean room count queries,
gated by the budget and by a k-anonymity suppression threshold), as used by the clean
room's query governance pipeline in Section 4.2.

In [3]:
from datetime import date
from typing import Optional
import numpy as np


class PrivacyBudgetExceededError(Exception):
    """Raised when a query would exceed a party's remaining daily epsilon budget."""


class PrivacyBudget:
    """Tracks one party's cumulative differential-privacy spend against a
    daily epsilon cap, resetting at midnight (local clean room time).
    """

    def __init__(self, party_id: str, daily_limit: float):
        self.party_id = party_id
        self.daily_limit = daily_limit
        self._spent_by_day: dict[date, float] = {}

    def remaining(self, today: Optional[date] = None) -> float:
        """Epsilon remaining for `today` (defaults to the current date)."""
        today = today or date.today()
        spent = self._spent_by_day.get(today, 0.0)
        return max(0.0, self.daily_limit - spent)

    def can_answer(self, epsilon_cost: float, today: Optional[date] = None) -> bool:
        """Returns True if a query costing `epsilon_cost` still fits today's budget."""
        if epsilon_cost <= 0:
            raise ValueError("epsilon_cost must be positive")
        return epsilon_cost <= self.remaining(today)

    def consume(self, epsilon_cost: float, today: Optional[date] = None) -> float:
        """Deducts `epsilon_cost` from today's budget and returns what remains.

        Raises PrivacyBudgetExceededError, with no partial spend recorded,
        if the query would overdraw the daily cap.
        """
        today = today or date.today()
        if not self.can_answer(epsilon_cost, today):
            raise PrivacyBudgetExceededError(
                f"party={self.party_id} requested epsilon={epsilon_cost:.3f}, "
                f"only {self.remaining(today):.3f} of {self.daily_limit:.3f} "
                f"remains today"
            )
        spent = self._spent_by_day.get(today, 0.0)
        self._spent_by_day[today] = spent + epsilon_cost
        return self.remaining(today)


class DPNoiseAdder:
    """Applies Laplace-mechanism noise (Equation 13.1) to clean room count
    queries, gated by a PrivacyBudget so no output is released without
    available epsilon.
    """

    def __init__(self, budget: PrivacyBudget, rng: Optional[np.random.Generator] = None):
        self.budget = budget
        self.rng = rng or np.random.default_rng()

    def answer_count_query(
        self,
        true_count: int,
        epsilon_cost: float,
        sensitivity: float = 1.0,
        k_anonymity_threshold: int = 100,
    ) -> Optional[float]:
        """Returns a noised count for a clean room aggregation query, or
        None if the query is suppressed for falling below the k-anonymity
        threshold (Table in Section 4.2: "Aggregation minimum").

        Suppressed queries consume no epsilon, since no output is released
        and therefore no privacy loss occurs.
        """
        if true_count < k_anonymity_threshold:
            return None
        self.budget.consume(epsilon_cost)
        scale = sensitivity / epsilon_cost
        noise = self.rng.laplace(loc=0.0, scale=scale)
        return true_count + noise

A representative advertiser session against a daily epsilon=1.0 budget illustrates
the intended usage (Test 2a below re-runs the same scenario with assertions for
stricter validation).

In [4]:
budget = PrivacyBudget(party_id="adv_12345", daily_limit=1.0)
noise_adder = DPNoiseAdder(budget, rng=np.random.default_rng(42))

queries = [
    {"query_id": "q1", "epsilon": 0.10, "true_count": 4200},
    {"query_id": "q2", "epsilon": 0.25, "true_count": 1830},
    {"query_id": "q3", "epsilon": 0.15, "true_count": 512},
]

for q in queries:
    noised = noise_adder.answer_count_query(q["true_count"], q["epsilon"])
    print(f"{q['query_id']}: noised_count={noised:.1f}, "
          f"epsilon_remaining={budget.remaining():.2f}")

# q4 would push cumulative spend to 0.5 + 0.6 = 1.1, over the 1.0 daily limit
try:
    noise_adder.answer_count_query(true_count=900, epsilon_cost=0.6)
except PrivacyBudgetExceededError as e:
    print(f"q4 blocked: {e}")

# Suppression: a group of 42 falls below the k=100 anonymity threshold
suppressed = noise_adder.answer_count_query(true_count=42, epsilon_cost=0.05)
print(f"suppressed query result (below k-anonymity threshold): {suppressed}")
print(f"epsilon_remaining after suppression (unchanged): {budget.remaining():.2f}")

q1: noised_count=4207.9, epsilon_remaining=0.90
q2: noised_count=1829.5, epsilon_remaining=0.65
q3: noised_count=520.4, epsilon_remaining=0.50
q4 blocked: party=adv_12345 requested epsilon=0.600, only 0.500 of 1.000 remains today
suppressed query result (below k-anonymity threshold): None
epsilon_remaining after suppression (unchanged): 0.50


**Test 2a.** A sequence of clean room queries from a single advertiser against a
daily epsilon=1.0 budget: three queries succeed and consume epsilon, a fourth is
blocked for exceeding the remaining budget, a fifth is suppressed for falling below
the k-anonymity threshold (and correctly consumes no epsilon).

In [5]:
# --- Test 2a: advertiser query sequence against a daily epsilon=1.0 budget ---
budget = PrivacyBudget(party_id="adv_12345", daily_limit=1.0)
noise_adder = DPNoiseAdder(budget, rng=np.random.default_rng(42))

queries = [
    {"query_id": "q1", "epsilon": 0.10, "true_count": 4200},
    {"query_id": "q2", "epsilon": 0.25, "true_count": 1830},
    {"query_id": "q3", "epsilon": 0.15, "true_count": 512},
]

for q in queries:
    noised = noise_adder.answer_count_query(q["true_count"], q["epsilon"])
    assert noised is not None
    print(f"{q['query_id']}: noised_count={noised:.1f}, "
          f"epsilon_remaining={budget.remaining():.2f}")

assert abs(budget.remaining() - 0.5) < 1e-9

# q4 would push cumulative spend to 0.5 + 0.6 = 1.1, over the 1.0 daily limit
try:
    noise_adder.answer_count_query(true_count=900, epsilon_cost=0.6)
    raise AssertionError("expected PrivacyBudgetExceededError")
except PrivacyBudgetExceededError as e:
    print(f"q4 blocked: {e}")

# Suppression: a group of 42 falls below the k=100 anonymity threshold
suppressed = noise_adder.answer_count_query(true_count=42, epsilon_cost=0.05)
assert suppressed is None
print(f"suppressed query result (below k-anonymity threshold): {suppressed}")
print(f"epsilon_remaining after suppression (unchanged): {budget.remaining():.2f}")
assert abs(budget.remaining() - 0.5) < 1e-9

# can_answer as a pre-check before consuming
proposed_epsilon = 0.6
print(f"can_answer({proposed_epsilon}) = {budget.can_answer(proposed_epsilon)}")
assert budget.can_answer(proposed_epsilon) is False
print("PASS: Test 2a")

q1: noised_count=4207.9, epsilon_remaining=0.90
q2: noised_count=1829.5, epsilon_remaining=0.65
q3: noised_count=520.4, epsilon_remaining=0.50
q4 blocked: party=adv_12345 requested epsilon=0.600, only 0.500 of 1.000 remains today
suppressed query result (below k-anonymity threshold): None
epsilon_remaining after suppression (unchanged): 0.50
can_answer(0.6) = False
PASS: Test 2a


**Test 2b.** Statistical check that `DPNoiseAdder`'s released noise matches
Equation 13.1's calibration: for a fixed epsilon and sensitivity, the empirical
variance across many trials should converge to the Laplace distribution's
theoretical variance, $2(\Delta f / \varepsilon)^2$.

In [6]:
# --- Test 2b: does the noise match Equation 13.1's sensitivity/epsilon calibration? ---
epsilon = 0.5
sensitivity = 1.0
n_trials = 5000
trial_budget = PrivacyBudget(party_id="test_party", daily_limit=n_trials * epsilon + 1)
trial_adder = DPNoiseAdder(trial_budget, rng=np.random.default_rng(7))

true_count = 10_000
samples = np.array([
    trial_adder.answer_count_query(
        true_count, epsilon_cost=epsilon, k_anonymity_threshold=0
    ) - true_count
    for _ in range(n_trials)
])
empirical_var = samples.var()
theoretical_var = 2 * (sensitivity / epsilon) ** 2
relative_error = abs(empirical_var - theoretical_var) / theoretical_var
print(f"empirical variance={empirical_var:.2f}, theoretical variance={theoretical_var:.2f}, "
      f"relative_error={relative_error:.2%}")
assert relative_error < 0.15, "Laplace noise scale does not match sensitivity/epsilon"
print("PASS: Test 2b - Laplace noise scale matches Equation 13.1's calibration")

empirical variance=7.74, theoretical variance=8.00, relative_error=3.31%
PASS: Test 2b - Laplace noise scale matches Equation 13.1's calibration
